# FIJI Processing

# TO DO 
Add description for preprocessing and obj file conversion. 
Here we import the .obj file, use Quadric Edge Collapse Decimation to halve the face number of the mesh, Laplacian Smooth to smooth the surface of the mesh, and export the mesh. With the exported mesh, we can do additional cleanup in Meshmixer. 

* Quadric Edge Collapse Decimation is a well-used method to simplify a mesh while preserving boundary and normals. 

In [ ]:
# We use the library pymeshlab for mesh processing  
import tifffile as tf
import cv2
import pymeshlab
import numpy as np
import matplotlib.pyplot as plt
from skimage import measure
from skimage.morphology import binary_erosion, convex_hull_image, area_opening
from skimage.transform import downscale_local_mean
from skimage import io, transform
#import trimesh
import os
from tkinter import filedialog
from pathlib import Path
import imagej
#import xarray

In [ ]:
# Initialize FIJI 
ij = imagej.init('sc.fiji:fiji', mode='interactive')
print(f"ImageJ2 version: {ij.getVersion()}")

In [ ]:
# Load the .tif file with the path 
file_path = filedialog.askopenfilename(initialdir = os.sep,title = "Select input TIF file", filetypes = (("TIFF Files","*.tif"),("all files","*.*")))
output_folder = Path(file_path).parent # Output folder for all images

# load test image
img_stack = tf.imread(file_path) # xarray.DataArray (Python)
#dataset = ij.io().open(file_path) # ImageJ image (Java)
dataset = ij.IJ.openImage(file_path) # ImageJ image (Java)

#Load the .obj file with the path 
#obj_file = filedialog.askopenfilename(initialdir = os.sep,title = "Select file", filetypes = (("Wavefront OBJ","*.obj"),("all files","*.*")))

# Set the resolution of the image stack -- this is from the microscope! 
xy_resolution = 1.625
z_resolution = 3

In [ ]:
# Image infos
def dump_info(image):
    """A handy function to print details of an image object."""
    name = image.name if hasattr(image, 'name') else None # xarray
    if name is None and hasattr(image, 'getName'): name = image.getName() # Dataset
    if name is None and hasattr(image, 'getTitle'): name = image.getTitle() # ImagePlus
    print(f" name: {name or 'N/A'}")
    print(f" type: {type(image)}")
    print(f"dtype: {image.dtype if hasattr(image, 'dtype') else 'N/A'}")
    print(f"shape: {image.shape}")
    print(f" dims: {image.dims if hasattr(image, 'dims') else 'N/A'}")

dump_info(dataset)

In [ ]:
# Show a single slice of yhe image
middle_slice = dataset.shape[2] / 2 # image middle slice

ij.py.show(img_stack[:,:,int(middle_slice)])

In [ ]:
# Read FIJI Macro
macro_path = filedialog.askopenfilename(initialdir = os.sep,title = "Select FIJI macro file", filetypes = (("Macro Files","*.ijm"),("all files","*.*")))
macro_file = open(macro_path, "r")

add = """
#@ String img_path
#@output Object image_title
#@output Object image_dir


"""
macro_file = add + macro_file.read()
#print(macro_file)

In [ ]:
# Launch FIJI Macro
args = {
    'img_path': file_path
}

result = ij.py.run_macro(macro_file, args)

In [ ]:
# Print Output
print(result.getOutput('image_dir'))
print(result.getOutput('image_title'))

* Output image name : **pyimagej.ome.tif**

### **Optional : Size Opening in binary image**

With Python : 
---

TOO long in time. But if you have a lot of time to lost, then you can run the cells below

In [ ]:
###### To replace FIJI Size Opening plugin ######
current_image = ij.WindowManager.getCurrentImage()
python_image = ij.py.from_java(current_image)
mask = area_opening(python_image.to_masked_array(), area_threshold=5000) # It's VERY long => Do overnight or better in the weekend
tf.imsave(result.getOutput('image_title')+"pyimagej_sizeOpening.ome.tif",mask)

In [ ]:
# Show a single slice of yhe image
ij.py.show(python_image[:,:,int(middle_slice)])

With Fiji : 
---

1. Open Fiji app
2. Open pyimagej.ome.tif (the previous output image)
3. Run Plugin > MorphoLibJ > Binary images > Size Opening 2D/3D (Min Voxel Number : 5000)

Step 1: Size opening in 2D slices
Often, a problem that we have is that the end tips connect with an adjacent slice. To minimize this problem, we will run size opening on individual 2D slices and remove noise and the final layer of an end tip in Z.

# TO DO 
Add diagram to show idea. 

**FOR BIG LUNG SAMPLE - Size Opening in 2D Slice**
---

In [ ]:
current_image = ij.WindowManager.getCurrentImage()
python_image = ij.py.from_java(current_image)

In [ ]:
### Size opening in 2D slice ###
# First, find the total number of slices. 
z = python_image.shape[0]

# Define a size to use Size Opening on 
min_size = 100 
# eg. for min_size = 100, I will remove all labels under 100 pixels in size in 2D 

with tf.TiffWriter(result.getOutput('image_title')+"pyimagej.ome.tif") as tif:
    img_number = 1
    for im in range(z): 
        # Look at each slice in skeleton stack
        img_slice = img_stack[im, :, :]
        num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(img_slice, connectivity=8)
        # Create a mask to filter labels based on size
        mask = np.zeros_like(img_slice, dtype=np.uint8)
        for label in range(1, num_labels):
            if stats[label, cv2.CC_STAT_AREA] >= min_size:
                mask[labels == label] = 255
        # Make a file name for the new img_slice 
        filename = f"size_opened_{im}" 
        # save the new skeleton 
        tif.write(mask, description = filename, metadata=None)
        img_number +=1 

After size opening step, time to **manual correction (sadly) => Go on FIJI**